# Tree Visualization

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

## Load tree metrics into DataFrame

In [ ]:
from viz.utils import PROJECT_ROOT
df = pd.read_csv(os.path.join(PROJECT_ROOT, "results/sim/tree/tree_summary.tsv"), sep="\t")
df.describe()

## Generate number of species and mean branch length pairs 

In [ ]:
def harmonic_number(n):
    return sum(1.0 / k for k in range(1, n + 1))

for expected_depth in [0.8, 1, 1.2]: 
    for species in [8, 16, 32, 64]:
        m = expected_depth / (2*harmonic_number(species)-2)
        print(f"        - {{mean: {m}, species: {species}}}") 

## Plot expected average distance to root

In [ ]:
def plot_scatter(ax, df, xcol, ycol, title):
    sns.scatterplot( data=df, x=xcol, y=ycol, hue='species', palette='tab10', ax=ax, s=40, alpha=0.6, legend=False )
    sns.regplot( data=df, x=xcol, y=ycol, scatter=False, ax=ax, color='black', ci=None )
    mn = min(df[xcol].min(), 0)
    mx = max(df[ycol].max(), 1)
    ax.plot([mn, mx], [mn, mx], "--", color='gray')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(title)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)

def plot_ratio(ax, df, ratio_col, title, bins=60):
    vals = df[ratio_col].replace([np.inf, -np.inf], np.nan).dropna()
    sns.histplot(vals, bins=bins, kde=True, ax=ax)
    ax.axvline(1.0, color='red', linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Observed / Expected')

fig, axes = plt.subplots(2, 2, figsize=(8, 6))

_df = df.copy()
_df['harmonic'] = _df['species'].apply(harmonic_number)

eps = 1e-12

_df['expected_steps'] = 2 * _df['harmonic'] - 2
_df['expected_dist'] = (2 * (_df['harmonic'] - 1) * _df['mean']).round(4)


_df['ratio_steps'] = _df['avg_leaf_steps_to_root'] / (_df['expected_steps'] + eps)
_df['ratio_dist'] = _df['avg_leaf_dist_to_root'] / (_df['expected_dist'] + eps)


plot_scatter( axes[0, 0], _df, 'expected_steps', 'avg_leaf_steps_to_root', 'Steps: Observed vs Expected' )
plot_scatter( axes[1, 0], _df, 'expected_dist', 'avg_leaf_dist_to_root', 'Distance: Observed vs Expected' )

plot_ratio( axes[0, 1], _df, 'ratio_steps', 'Steps: Ratio Distribution' )
plot_ratio( axes[1, 1], _df, 'ratio_dist', 'Distance: Ratio Distribution' )

plt.tight_layout()

## Yule tree expected max height
### Topological distance

In [ ]:
# step distance
def exact_yule_height_distribution(n_max):
    # dists[n][h] = Probability that a tree with n nodes has height h
    # We use a dictionary or list of arrays to store distributions
    dists = {0: np.array([1.0])}  # Tree with 0 nodes has height 0 with prob 1
    
    for n_ in range(1, n_max + 1):
        # The new distribution for n nodes
        # Max possible height is n, so we need an array of size n+1
        current_dist = np.zeros(n_ + 1)
        
        # In a Random BST / Yule tree, the root splits the remaining 
        # n-1 nodes into i and (n-1-i) with probability 1/n each.
        for i in range(n_):
            left_dist = dists[i]
            right_dist = dists[n_ - 1 - i]
            
            # Combine distributions: H_n = 1 + max(H_left, H_right)
            for h_l, p_l in enumerate(left_dist):
                for h_r, p_r in enumerate(right_dist):
                    h_max = max(h_l, h_r)
                    # We add 1 because of the edge from the new root
                    current_dist[h_max + 1] += (p_l * p_r) / n_
                    
        dists[n_] = current_dist
    return dists

In [ ]:
all_dists = exact_yule_height_distribution(70)

In [ ]:
def exact_yule_height_in_steps(_, n):
    internal_nodes = n - 1  # For n leaves, there are n-1 internal nodes in a binary tree
    dist_n = all_dists[internal_nodes]
    # Calculate Expected Value: E[H] = sum(h * P(h))
    expected_height = sum(h * p for h, p in enumerate(dist_n))
    return expected_height

### Distance including branch lengths

In [ ]:
from scipy.signal import fftconvolve

def precalculate_yule_cdfs_ensemble(n_max, dx=0.005, max_x=20.0):
    """
    Precalculates the exact CDF of the maximum leaf-to-root distance
    over the ENTIRE ensemble of Yule topologies of size up to n_max leaves.
    """
    mean_length=1.0 # since we can scale later
    x_bins = np.arange(0, max_x, dx)
    num_bins = len(x_bins)
    
    # cdfs[M] stores the CDF of the max distance for a tree with M internal nodes.
    # Base case: M = 0 (1 leaf). Max distance to root is exactly 0.
    # P(H_0 <= x) = 1.0 for all x >= 0.
    cdfs = {0: np.ones(num_bins)}
    
    # Exact probability mass of an exponential branch in each bin:
    # P(i*dx <= L < (i+1)*dx)
    pdf_L = np.exp(-x_bins / mean_length) * (1.0 - np.exp(-dx / mean_length))

    # A tree with n leaves has M = n - 1 internal nodes
    for M in range(1, n_max):
        # 1. Compute G_i(x) = CDF(L + H_i) for all subtrees i < M
        g_cdfs = {}
        for i in range(M):
            conv = fftconvolve(pdf_L, cdfs[i], mode='full')[:num_bins]
            # Exact boundary alignment: Shift right by 1 bin because at x=0, 
            # the probability of (L + H_i <= 0) is 0.
            g_cdfs[i] = np.zeros(num_bins)
            g_cdfs[i][1:] = conv[:-1]
        
        # 2. Average over all possible Yule splits
        avg_cdf = np.zeros(num_bins)
        for i in range(M):
            avg_cdf += g_cdfs[i] * g_cdfs[M - 1 - i]
        avg_cdf /= M
        
        # 3. Clip to stabilize numeric precision
        cdfs[M] = np.clip(avg_cdf, 0.0, 1.0)
        
    return x_bins, cdfs

In [ ]:
# Precalculate once for up to 75 leaves (to support n=70)
X_BINS, ALL_CDFS = precalculate_yule_cdfs_ensemble(n_max=75, dx=0.0005, max_x=20.0)

In [ ]:
def exact_yule_distance(mean, n):
    if n <= 1:
        return 0.0

    n_internal_nodes = n - 1
    cdf = ALL_CDFS[n_internal_nodes]
    dx = X_BINS[1] - X_BINS[0]
    
    # E[X] = integral(1 - CDF)
    expected_val_unit = np.sum(1.0 - cdf) * dx
    return expected_val_unit * mean

## Yule tree average leaf distance and expected average pairwise leaf distance

In [ ]:
def expected_average_leaf_distance(mean, species):
    return mean * (2.0 * harmonic_number(species) - 2.0)

def expected_pairwise_leaf_step_distance(_, species):
    return (4.0 * (species+1.0) / (species-1.0) * (harmonic_number(species)-1.0) - 4.0) 

def expected_pairwise_leaf_distance(mean, species):
    return (4.0 * (species+1.0) / (species-1.0) * (harmonic_number(species)-1.0) - 4.0) * mean

## Plot measures stratified by group and expected leaf to root distance

In [ ]:
def compare_measures_across_groups(df, observed_col, expected_func=None):
    groups = {}
    for row in df.itertuples(index=False):
        mean = row.mean
        species = row.species
        # right now the species and branch means are such that the expected average leaf distance is exactly the same for all groups
        key = expected_average_leaf_distance(mean, species)
        groups.setdefault(round(key, 4), set()).add((mean, species))
    groups = list(groups.values())

    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(8, 10))
    bins = 40

    species_list = sorted(df["species"].unique())
    palette = sns.color_palette("tab10", len(species_list))
    color_map = dict(zip(species_list, palette))

    for ax, group in zip(axes, groups):
        # IMPORTANT: enforce deterministic draw order
        group_sorted = sorted(group, key=lambda x: x[1])  # sort by species
        for (m, s) in group_sorted:
            subset = df[(df["mean"] == m) & (df["species"] == s)]
            if subset.empty:
                continue

            sns.histplot(subset[observed_col], ax=ax, color=color_map[s], label=f"species={s}, mean={m}", stat="density", bins=bins, binrange=(0, df[observed_col].max()), alpha=0.5, kde=True, )
            empirical_mean = subset[observed_col].mean()
            ax.axvline(empirical_mean, color=color_map[s], linestyle=":", linewidth=1.5)
            if expected_func:
                expected_value = expected_func(m, s)
                ax.axvline(expected_value, color=color_map[s], linestyle="--", linewidth=1.5)

        # FIX: stable legend order (same as drawing order)
        handles, labels = ax.get_legend_handles_labels()
        # sort legend by species extracted from label
        def extract_species(label):
            return int(label.split("species=")[1].split(",")[0])

        sorted_legend = sorted(zip(handles, labels), key=lambda hl: extract_species(hl[1]))
        handles, labels = zip(*sorted_legend)

        expectation = m * (2.0 * harmonic_number(s) - 2.0)
        ax.legend(handles, labels, title=f"expectated leaf dist {expectation:.2f}")
        ax.set_ylabel("Density")

    plt.xlabel(observed_col)
    plt.tight_layout()

#compare_measures_across_groups(df, 'avg_leaf_steps_to_root')
#compare_measures_across_groups(df, 'avg_leaf_dist_to_root')  # the expectation is in the title of the plots, no need to draw it

#compare_measures_across_groups(df, 'var_leaf_steps_to_root') # here the 64 species has more variance now since now we have more branching points to place and there more variance in that 
#compare_measures_across_groups(df, 'var_leaf_dist_to_root') # the 8 species has more variance perhaps because here the variance of blanch lengths has a larger impact
plt.show()
#compare_measures_across_groups(df, 'max_leaf_steps_to_root', exact_yule_height_in_steps)
#compare_measures_across_groups(df, 'max_leaf_dist_to_root', exact_yule_distance) # we cannot really use a analytical expectation here

compare_measures_across_groups(df, 'avg_pairwise_leaf_step_distance', expected_pairwise_leaf_step_distance)
#compare_measures_across_groups(df, 'colless_index')